# 08 — Fine-Tune XLM-RoBERTa

## Goal
Fine-tune a multilingual transformer to classify Sinhala and English farming questions.

**What happens using this code?** The pretrained `FacebookAI/xlm-roberta-base` model is fine-tuned on the training split, and the validation split is used to choose the best checkpoint. This notebook does not use the test set for model selection.

**Hardware note:** this is a heavier experiment than TF-IDF. If GPU memory is limited, keep batch sizes small or start with one epoch.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "datasets/processed"
train_df = pd.read_csv(DATA / "paddy_train.csv", keep_default_na=False, encoding="utf-8-sig")
val_df = pd.read_csv(DATA / "paddy_validation.csv", keep_default_na=False, encoding="utf-8-sig")
labels = sorted(train_df["label"].unique().tolist())
label2id = {v:i for i,v in enumerate(labels)}
id2label = {i:v for v,i in label2id.items()}
assert set(val_df["label"]).issubset(set(labels)), "Validation contains a category missing from training."

MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LENGTH = 160

def to_dataset(frame):
    data = frame[["question", "label"]].copy()
    data["label"] = data["label"].map(label2id).astype(int)
    return Dataset.from_pandas(data, preserve_index=False)

train_ds, val_ds = to_dataset(train_df), to_dataset(val_df)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch["question"], truncation=True, max_length=MAX_LENGTH)
train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(labels), label2id=label2id, id2label=id2label)

def compute_metrics(eval_pred):
    logits, gold = eval_pred
    pred = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(gold, pred, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(gold, pred), "precision_macro": p, "recall_macro": r, "f1_macro": f1}

out_dir = ROOT / "models/problem_classifier/xlm_roberta_base"
args = TrainingArguments(
    output_dir=str(out_dir),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
    seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tokenizer, data_collator=collator, compute_metrics=compute_metrics)
trainer.train()
val_output = trainer.predict(val_ds)
val_pred_ids = np.argmax(val_output.predictions, axis=1)
val_pred = [id2label[int(i)] for i in val_pred_ids]
metrics = compute_metrics((val_output.predictions, val_output.label_ids))
metrics = {k: float(v) for k,v in metrics.items()}
print("Validation metrics:", json.dumps(metrics, indent=2))
trainer.save_model(str(out_dir / "best_model"))
tokenizer.save_pretrained(str(out_dir / "best_model"))
(ROOT / "models/problem_classifier/label_map_xlm_roberta.json").write_text(json.dumps({"label2id":label2id,"id2label":{str(k):v for k,v in id2label.items()}}, ensure_ascii=False, indent=2), encoding="utf-8")
(ROOT / "models/model_evaluation_results/xlmr_validation_metrics.json").write_text(json.dumps({"model":"xlm_roberta_base", **metrics}, indent=2), encoding="utf-8")
pd.DataFrame({"question":val_df["question"], "language":val_df["language"], "issue_id":val_df["issue_id"], "true_label":val_df["label"], "predicted_label":val_pred}).to_csv(ROOT / "models/model_evaluation_results/xlmr_validation_predictions.csv", index=False, encoding="utf-8-sig")
print("Saved best transformer and validation results.")

**Model choice:** XLM-RoBERTa is the main multilingual transformer candidate. It is not guaranteed to be more accurate than the baseline on this particular dataset; use the validation comparison in notebook 09.